# ML Challenge 2026 — final: DuckDB vs embeddings vs blends, best one submitted (CPU)

Inputs (Add Input): competition data; the v9 output dataset (work/lgbm.txt ...); the original
embeddings (ann/ann_train.parquet ...) and/or the fine-tuned ones (ann_ft/ann_train.parquet ...).

Every variant is scored by the same 5-fold CV on the held-out entities (the main run never
trained on them, the fine-tuning excluded them):
  duckdb      the current submission's rule (second-stage model + exact expected F0.5)
  A_<emb>     DuckDB candidates re-scored with that embedding's similarity/rank added
  B_<emb>     A + that embedding's top-10 records DuckDB never proposed
  E_<emb>     embeddings alone (their top-10 per S1, no DuckDB)
A variant replaces "duckdb" only if it is better on average AND in at least 4 of 5 folds.
One submission is written to /kaggle/working/final/ and validated after memory is freed.

In [ ]:
import gc, glob, os, shutil, subprocess, sys, time
from pathlib import Path

import duckdb, lightgbm as lgb, numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq

T_START = time.perf_counter()
DATA_DIR = Path("/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/amazon-ml-challenge-2026/dataset")
_lgbm = glob.glob("/kaggle/input/**/work/lgbm.txt", recursive=True)
assert _lgbm, "attach the v9 output dataset (it must contain work/lgbm.txt)"
WORK = Path(_lgbm[0]).parent
EMB = {}
for f in glob.glob("/kaggle/input/**/ann_train.parquet", recursive=True):
    d = Path(f).parent
    if (d / "ann_test.parquet").exists():
        EMB.setdefault("ft" if d.name == "ann_ft" else "orig", d)
FINAL = Path("/kaggle/working/final")
TMP = Path("/kaggle/working/tmp")
for d in (FINAL, TMP):
    d.mkdir(parents=True, exist_ok=True)
GT_PATH = DATA_DIR / "train" / "train_ground_truth.tsv"
PROB_FLOOR, ANN_K = 0.01, 10
SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
print(f"v9 run: {WORK}")
for k, d in EMB.items():
    print(f"embeddings '{k}': {d}")
if not EMB:
    print("no embeddings attached: only the DuckDB rule will be used")

def lap(label):
    print(f"  [{label}: {(time.perf_counter() - T_START) / 60:.1f} min elapsed]")

def duplicate_records(folder):
    """Records given to more than one S1. The truth never does this, so each one is a wrong match."""
    seen, dup = set(), 0
    with open(folder / "matching_results.tsv") as fh:
        next(fh)
        for line in fh:
            for c in line.rstrip("\n").split("\t")[1].split(","):
                if c:
                    dup += c in seen
                    seen.add(c)
    return dup

def validate(folder):
    r = subprocess.run([sys.executable, str(DATA_DIR.parent / "validate_submission.py"),
                        "--matching", str(folder / "matching_results.tsv"),
                        "--candidate", str(folder / "candidate_pairs.tsv"),
                        "--test-dir", str(DATA_DIR / "test")], capture_output=True, text=True)
    print(r.stdout[-2500:] or r.stderr[-2000:])
    print(f"{folder.name}: exit code {r.returncode}  ->  {'PASS' if r.returncode == 0 else 'FAIL'}")
    return r.returncode == 0

## candidate_pairs.tsv (first, while memory is empty)

In [ ]:
def build_candidate_pairs(out_path):
    tmp = TMP / "cand"
    tmp.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.execute(f"SET memory_limit='12GB'; SET temp_directory='{tmp}'; SET preserve_insertion_order=false")
    aggs = []
    for f in sorted(WORK.glob("test_candidates*.parquet")):
        a = tmp / f"agg_{f.stem}.parquet"
        con.execute(f"""COPY (SELECT s1_id, string_agg(cand_id, ',' ORDER BY cand_id) AS c
                             FROM read_parquet('{f}') GROUP BY s1_id) TO '{a}' (FORMAT PARQUET)""")
        aggs.append(str(a))
    ids = pd.read_csv(DATA_DIR / "test" / "test_source1.tsv", sep="\t", header=0, names=SOURCE_COLUMNS,
                      usecols=["entity_id"], dtype=str)["entity_id"]
    con.register("s1ord", pa.table({"entity_id": pa.array(ids.tolist(), pa.string()),
                                    "i": pa.array(np.arange(len(ids)), pa.int64())}))
    # an entity without candidates gets NULL, which the CSV writer leaves as an empty field
    con.execute(f"""COPY (SELECT s.entity_id AS source1_entity_id, a.c AS candidate_entity_ids
                          FROM s1ord s LEFT JOIN read_parquet({aggs}) a ON a.s1_id = s.entity_id
                          ORDER BY s.i)
                    TO '{out_path}' (FORMAT CSV, DELIMITER '\t', HEADER)""")
    con.close()
    shutil.rmtree(tmp, ignore_errors=True)
    return len(aggs)

n_shards = build_candidate_pairs(FINAL / "candidate_pairs.tsv")
print(f"candidate_pairs.tsv from {n_shards} shards")
lap("candidates")

## Selection rules and a fast exclusivity repair

In [ ]:
def f05(pred, true):
    k, m = len(pred), len(true)
    if k == 0 and m == 0:
        return 1.0
    if k == 0 or m == 0:
        return 0.0
    h = len(pred & true)
    return 0.0 if h == 0 else 1.25 * h / (k + 0.25 * m)

def macro_f05(pred, truth):
    return sum(f05(pred.get(s, set()), t) for s, t in truth.items()) / len(truth)

def select_base(df, e_m):
    """The main notebook's rule: cut at the k maximizing 1.25*sum(p_1..k) / (k + 0.25*E[m])."""
    d = df.sort_values(["s1_id", "p"], ascending=[True, False], kind="stable")
    p = d["p"].to_numpy(np.float64)
    g = d.groupby("s1_id", sort=False)
    k = g.cumcount().to_numpy() + 1
    em = d["s1_id"].map(e_m).fillna(0.0).to_numpy(np.float64)
    d = d.assign(score=1.25 * g["p"].cumsum().to_numpy() / (k + 0.25 * em), k=k,
                 log1mp=np.log1p(-np.clip(p, 0, 1 - 1e-12)))
    best = d.loc[d.groupby("s1_id", sort=False)["score"].idxmax(), ["s1_id", "score", "k"]].set_index("s1_id")
    score0 = np.exp(d.groupby("s1_id", sort=False)["log1mp"].sum())
    kstar = best["k"].where(~(score0 > best["score"]), 0)
    sel = d[d["k"] <= d["s1_id"].map(kstar).to_numpy()]
    out = {s: set() for s in kstar.index}
    for s, c in zip(sel["s1_id"].to_numpy(), sel["cand_id"].to_numpy()):
        out[s].add(c)
    return out

def _poisson_binomial(ps):
    dist = np.ones(1)
    for q in ps:
        dist = np.convolve(dist, [1.0 - q, q])
    return dist

def best_cut(ps, cap=30):
    """Exact expected F0.5 of predicting the top k, for every k; returns the best k.
    Pairs are treated as independent; ps must be sorted descending."""
    ps = ps[:cap]
    n = len(ps)
    suffix = [None] * (n + 1)
    suffix[n] = np.ones(1)
    for i in range(n - 1, -1, -1):
        suffix[i] = np.convolve(suffix[i + 1], [1.0 - ps[i], ps[i]])
    best_k, best = 0, suffix[0][0]                 # k = 0 scores 1 only when there is no match
    prefix = np.ones(1)
    for k in range(1, n + 1):
        prefix = np.convolve(prefix, [1.0 - ps[k - 1], ps[k - 1]])
        h = np.arange(k + 1)[:, None]
        r = np.arange(n - k + 1)[None, :]
        e = prefix @ (1.25 * h / (k + 0.25 * (h + r))) @ suffix[k]
        if e > best:
            best_k, best = k, e
    return best_k

def select_exact(df, e_m=None):
    d = df.sort_values(["s1_id", "p"], ascending=[True, False], kind="stable")
    out = {}
    sids, cids, ps = d["s1_id"].to_numpy(), d["cand_id"].to_numpy(), d["p"].to_numpy(np.float64)
    starts = np.flatnonzero(np.r_[True, sids[1:] != sids[:-1]])
    ends = np.r_[starts[1:], len(sids)]
    for a, b in zip(starts, ends):
        k = best_cut(ps[a:b])
        out[sids[a]] = set(cids[a:a + k])
    return out

def repair_fast(pred, df, select, rounds=5):
    """A record can belong to one S1 only: keep it with the highest-probability claimant.
    Removals accumulate, and only the entities that lost a record are re-selected."""
    total = 0
    for _ in range(rounds):
        sel = pd.DataFrame([(s, c) for s, cs in pred.items() for c in cs], columns=["s1_id", "cand_id"])
        dup = sel[sel["cand_id"].duplicated(keep=False)]
        if dup.empty:
            break
        total += dup["cand_id"].nunique()
        d = dup.merge(df[["s1_id", "cand_id", "p"]], on=["s1_id", "cand_id"], how="left").fillna({"p": 0.0})
        d = d.sort_values(["cand_id", "p", "s1_id"], ascending=[True, False, True])
        losers = d[d.duplicated("cand_id", keep="first")]
        df = df[~pd.MultiIndex.from_arrays([df["s1_id"], df["cand_id"]]).isin(
            pd.MultiIndex.from_arrays([losers["s1_id"], losers["cand_id"]]))]
        affected = set(losers["s1_id"])
        redo = select(df[df["s1_id"].isin(affected)])
        for s in affected:
            pred[s] = redo.get(s, set())
    return pred, total

# the fast repair must agree with the slow one on a small random case
_rng = np.random.default_rng(1)
_df = pd.DataFrame({"s1_id": [f"s{i // 4}" for i in range(400)],
                    "cand_id": [f"c{x}" for x in _rng.integers(0, 150, 400)], "p": _rng.random(400)})
_df = _df.drop_duplicates(["s1_id", "cand_id"])
_fast, _ = repair_fast(select_exact(_df), _df, select_exact)
_seen = [c for cs in _fast.values() for c in cs]
assert len(_seen) == len(set(_seen)), "fast repair left a record with two owners"
assert best_cut(np.array([0.99])) == 1 and best_cut(np.array([0.05])) == 0
print("selection checks PASS")

## Held-out probabilities from the v9 model (calib + val folds)

In [ ]:
booster = lgb.Booster(model_file=str(WORK / "lgbm.txt"))
_cal = np.load(WORK / "calib.npz")
calibrate = lambda raw: np.interp(raw, _cal["x"], _cal["y"])   # = IsotonicRegression.predict (clip)
FEATURES = booster.feature_name()

def fold_of(s1_ids):                               # identical to the main notebook
    h = pd.util.hash_array(np.asarray(s1_ids), hash_key="0" * 16) % 100
    return np.where(h < 70, 0, np.where(h < 85, 1, 2))

t0 = time.perf_counter()
pf = pq.ParquetFile(WORK / "train_features.parquet")
parts = []
for batch in pf.iter_batches(batch_size=1_000_000, columns=["s1_id", "cand_id"] + FEATURES):
    b = batch.to_pandas()
    b = b[fold_of(b["s1_id"].to_numpy()) > 0]
    if len(b):
        p = calibrate(booster.predict(b[FEATURES]))
        parts.append(pd.DataFrame({"s1_id": b["s1_id"].to_numpy(), "cand_id": b["cand_id"].to_numpy(), "p": p}))
held = pd.concat(parts, ignore_index=True)
del parts

gt = pd.read_csv(DATA_DIR / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                 keep_default_na=False, na_values=[])
ents = set(held["s1_id"].unique())
truth = {s: (set(m.split(",")) if m else set())
         for s, m in zip(gt["source1_entity_id"], gt["matched_entity_ids"]) if s in ents}
pos = {(s, c) for s, cs in truth.items() for c in cs}
held["y"] = [int(k in pos) for k in zip(held["s1_id"].to_numpy(), held["cand_id"].to_numpy())]

# mimic the test files: E[m] from all candidates, pairs kept only above the floor
held_em = held.groupby("s1_id")["p"].sum()
held = held[held["p"] >= PROB_FLOOR].reset_index(drop=True)
print(f"held-out entities {len(truth):,}  pairs above floor {len(held):,}  "
      f"positives {held['y'].sum():,}  ({time.perf_counter() - t0:.0f}s)")

lap("held-out")

## Features for every variant

In [ ]:
S2_FEATURES = ["p", "rank", "n", "e_m", "p_max", "p_2nd", "p_rel", "gap_top", "gap_prev",
               "gap_next", "cum_p", "rest_p", "n_over_half", "logit_p"]
ANN_FEATURES = ["ann_score", "ann_rank", "in_duck", "ann_top", "ann_gap"]
S2_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=15, min_data_in_leaf=200,
                 feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                 seed=7, deterministic=True, verbosity=-1, num_threads=4)
S2_ROUNDS = 300

def s2_features(df, e_m):
    d = df.sort_values(["s1_id", "p"], ascending=[True, False], kind="stable").reset_index(drop=True)
    g = d.groupby("s1_id", sort=False)["p"]
    d["rank"] = g.cumcount() + 1
    d["n"] = g.transform("size")
    d["e_m"] = d["s1_id"].map(e_m).fillna(0.0)
    d["p_max"] = g.transform("max")
    d["p_2nd"] = d["s1_id"].map(d.loc[d["rank"] == 2].set_index("s1_id")["p"]).fillna(0.0)
    d["p_rel"] = d["p"] / d["p_max"].clip(lower=1e-9)
    d["gap_top"] = d["p_max"] - d["p"]
    d["gap_prev"] = (g.shift(1) - d["p"]).fillna(0.0)
    d["gap_next"] = (d["p"] - g.shift(-1)).fillna(d["p"])
    d["cum_p"] = g.cumsum()
    d["rest_p"] = d["e_m"] - d["p"]
    d["n_over_half"] = (d["p"] > 0.5).groupby(d["s1_id"], sort=False).transform("sum")
    d["logit_p"] = np.log(np.clip(d["p"], 1e-6, 1 - 1e-6) / np.clip(1 - d["p"], 1e-6, 1))
    for c in d.columns:
        if d[c].dtype == np.float64:
            d[c] = d[c].astype(np.float32)          # halves the memory of the feature table
    return d

def fit(d, feats):
    return lgb.train(S2_PARAMS, lgb.Dataset(d[feats], d["y"]), num_boost_round=S2_ROUNDS)

def with_ann(pairs, ann_path, cand_files, k):
    """DuckDB pairs + embedding score/rank (NaN outside the embedding list); embedding top-k
    records that were never DuckDB candidates (p = 0); and the embedding top-k on their own."""
    con = duckdb.connect()
    con.execute(f"SET memory_limit='10GB'; SET temp_directory='{TMP}'; SET preserve_insertion_order=false")
    con.register("pairs_in", pa.Table.from_pandas(pairs[["s1_id", "cand_id", "p"]], preserve_index=False))
    con.execute(f"""
        CREATE TABLE ents AS SELECT DISTINCT s1_id FROM pairs_in;
        CREATE TABLE annx AS SELECT a.s1_id, a.cand_id, a.ann_score, a.ann_rank
                             FROM read_parquet('{ann_path}') a JOIN ents USING (s1_id);
        CREATE TABLE top1 AS SELECT s1_id, MAX(ann_score) AS ann_top FROM annx GROUP BY 1;""")
    known = con.sql("""SELECT p.s1_id, p.cand_id, p.p, a.ann_score, a.ann_rank::FLOAT AS ann_rank, 1 AS in_duck, t.ann_top
                       FROM pairs_in p LEFT JOIN annx a USING (s1_id, cand_id) LEFT JOIN top1 t USING (s1_id)""").df()
    extra = con.sql(f"""SELECT a.s1_id, a.cand_id, 0.0 AS p, a.ann_score, a.ann_rank::FLOAT AS ann_rank, 0 AS in_duck, t.ann_top
                        FROM annx a ANTI JOIN read_parquet({[str(f) for f in cand_files]}) c USING (s1_id, cand_id)
                        JOIN top1 t USING (s1_id) WHERE a.ann_rank <= {k}""").df()
    alone = con.sql(f"""SELECT a.s1_id, a.cand_id, a.ann_score AS p, a.ann_score, a.ann_rank::FLOAT AS ann_rank,
                               0 AS in_duck, t.ann_top
                        FROM annx a JOIN top1 t USING (s1_id) WHERE a.ann_rank <= {k}""").df()
    con.close()
    for d in (known, extra, alone):
        d["ann_gap"] = d["ann_top"] - d["ann_score"]
    return known, extra, alone

def label(d):
    d["y"] = [int(k in pos) for k in zip(d["s1_id"].to_numpy(), d["cand_id"].to_numpy())]
    return d

VARIANTS = {"duckdb": (s2_features(held, held_em), S2_FEATURES)}
for name, d in EMB.items():
    known, extra, alone = with_ann(held, d / "ann_train.parquet", [WORK / "train_candidates.parquet"], ANN_K)
    known, extra, alone = label(known), label(extra), label(alone)
    print(f"  {name}: DuckDB pairs with an embedding score {known['ann_score'].notna().mean():.0%}; "
          f"embedding-only pairs {len(extra):,} ({extra['y'].sum():,} true)")
    VARIANTS[f"A_{name}"] = (s2_features(known, held_em), S2_FEATURES + ANN_FEATURES)
    VARIANTS[f"B_{name}"] = (s2_features(pd.concat([known, extra], ignore_index=True), held_em),
                             S2_FEATURES + ANN_FEATURES)
    VARIANTS[f"E_{name}"] = (s2_features(alone, alone.groupby("s1_id")["p"].sum()), S2_FEATURES + ANN_FEATURES)
    del known, extra, alone
gc.collect()
lap("features")

## 5-fold CV of every variant

In [ ]:
all_ents = np.array(sorted(truth))
cv_fold = pd.util.hash_array(all_ents, hash_key="1" * 16) % 5          # same folds as the selector
fold_of_ent = dict(zip(all_ents, cv_fold))
scores = {n: [] for n in ["base (v9 rule)"] + list(VARIANTS)}
for f in range(5):
    te_truth = {s: truth[s] for s in all_ents[cv_fold == f]}
    te_base = held[held["s1_id"].map(fold_of_ent) == f]
    scores["base (v9 rule)"].append(macro_f05(select_base(te_base, held_em), te_truth))
    for name, (d, feats) in VARIANTS.items():
        cv = d["s1_id"].map(fold_of_ent)
        tr, te = d[cv != f], d[cv == f]
        q = te[["s1_id", "cand_id"]].assign(p=fit(tr, feats).predict(te[feats]))
        scores[name].append(macro_f05(select_exact(q), te_truth))
ref = np.array(scores["duckdb"])
print(f"\n  {'variant':16s} {'macro F0.5':>10s} {'vs duckdb':>10s}   per fold")
for name, sc in scores.items():
    print(f"  {name:16s} {np.mean(sc):10.4f} {np.mean(sc) - ref.mean():+10.4f}   "
          + " ".join(f"{x:+.4f}" for x in np.array(sc) - ref))
wins = {n: np.mean(s) - ref.mean() for n, s in scores.items()
        if n != "duckdb" and np.mean(s) > ref.mean() and (np.array(s) > ref).sum() >= 4}
WINNER = max(wins, key=wins.get) if wins else "duckdb"
print(f"\nWINNER: {WINNER}" + (f"  (+{wins[WINNER]:.4f} vs duckdb on held-out entities)" if WINNER in wins else ""))
lap("cv")

## Apply the winner to test, write, free memory, validate

In [ ]:
if WINNER != "base (v9 rule)":
    WIN_D, WIN_FEATS = VARIANTS[WINNER]
    model = fit(WIN_D, WIN_FEATS)
    del WIN_D
del VARIANTS, held
gc.collect()

probs = pd.concat([pd.read_parquet(f) for f in sorted(WORK.glob("test_probs*.parquet"))], ignore_index=True)
sums = pd.concat([pd.read_parquet(f) for f in sorted(WORK.glob("test_sums*.parquet"))],
                 ignore_index=True).groupby("s1_id")["e_m"].sum()
extra_pairs = None
if WINNER == "base (v9 rule)":
    rows, em = probs, sums
elif WINNER == "duckdb":
    rows, em = probs, sums
else:
    emb = WINNER.split("_", 1)[1]
    known, extra, alone = with_ann(probs, EMB[emb] / "ann_test.parquet", sorted(WORK.glob("test_candidates*.parquet")), ANN_K)
    if WINNER.startswith("A"):
        rows, em = known, sums
    elif WINNER.startswith("B"):
        rows, em, extra_pairs = pd.concat([known, extra], ignore_index=True), sums, extra[["s1_id", "cand_id"]]
    else:
        rows, em, extra_pairs = alone, alone.groupby("s1_id")["p"].sum(), alone[["s1_id", "cand_id"]]
    del known, extra, alone
del probs
gc.collect()
print(f"test rows {len(rows):,}")
if WINNER == "base (v9 rule)":
    q, rule = rows[["s1_id", "cand_id", "p"]], (lambda df: select_base(df, em))
else:
    t = s2_features(rows, em)
    q, rule = t[["s1_id", "cand_id"]].assign(p=model.predict(t[WIN_FEATS])), select_exact
    del t
del rows
gc.collect()
lap("test scored")

pred = rule(q)
pred, contested = repair_fast(pred, q, rule)
print(f"  contested records resolved {contested:,}")
del q
gc.collect()
lap("test selected")

s1_ids = pd.read_csv(DATA_DIR / "test" / "test_source1.tsv", sep="\t", header=0, names=SOURCE_COLUMNS,
                     usecols=["entity_id"], dtype=str)["entity_id"].tolist()
with open(FINAL / "matching_results.tsv", "w") as fh:
    fh.write("source1_entity_id\tmatched_entity_ids\n")
    for s in s1_ids:
        fh.write(f"{s}\t{','.join(sorted(pred.get(s, ())))}\n")

if extra_pairs is not None:
    # a match the embeddings found must also be listed as a candidate
    chosen = {(s, c) for s, cs in pred.items() for c in cs}
    add = {}
    for s, c in zip(extra_pairs["s1_id"].to_numpy(), extra_pairs["cand_id"].to_numpy()):
        if (s, c) in chosen:
            add.setdefault(s, []).append(c)
    src_path = FINAL / "candidate_pairs.tsv"
    tmp_path = TMP / "candidate_pairs.tsv"
    with open(src_path) as src, open(tmp_path, "w") as dst:
        dst.write(next(src))
        for line in src:
            s, c = line.rstrip("\n").split("\t", 1)
            if s in add:
                have = set(filter(None, c.strip('"').split(",")))
                c = ",".join(sorted(have | set(add[s])))
            dst.write(f"{s}\t{c}\n")
    shutil.move(tmp_path, src_path)
    print(f"  added {sum(len(v) for v in add.values()):,} embedding-found matches to candidate_pairs.tsv")

k = np.array([len(pred.get(s, ())) for s in s1_ids])
print(f"  wrote {len(s1_ids):,} rows: empty {np.mean(k == 0):.1%}  mean set size {k.mean():.2f}")
del pred, extra_pairs, sums
gc.collect()                                # the validator needs the memory
validate(FINAL)
print(f"records assigned to more than one S1: {duplicate_records(FINAL):,}")
lap("done")
print(f"\nsubmit {FINAL}  (winner: {WINNER})")